In [2]:
import pymupdf  # PyMuPDF

def extract_text(pdf_path):
    doc = pymupdf.open(pdf_path)
    full_text = []
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        full_text.append(page.get_text())
        
    return "\n".join(full_text)

extracted_content = extract_text(r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\raw\phd\sop\2025_Fall_HyungjooChae_Statement_of_Purpose_GaTech (2).pdf")
print(extracted_content)

STATEMENT OF PURPOSE
Hyungjoo Chae
https://hyungjoo-homepage.netlify.app
Consider a developer asking an agent to create a simple personal website, or a user requesting it to order headphones
on Amazon. I still vividly remember the first time I saw a digital agent in action – how fascinatingly it independently
browses the internet, makes decisions, and completes a task. Whether it was a developer asking the agent to create a
simple personal website or an online shopper instructing it to order headphones on Amazon, I was captivated by the
agents’ ability to explore and interact with the digital world autonomously.
This initial awe gradually deepened as I began to see their broader potential – not just as passive tools, but as
transformative assistants capable of revolutionizing how humans interact with computers. Over time, my fascination
has evolved into the recognition of digital agents’ capacity to grow beyond basic task execution and become
intellectual partners, adept at understandi

In [3]:
from typing import List
from pydantic import BaseModel, Field

# Define the Pydantic schema for the PhD SOP extraction
class PhDScholarSOP(BaseModel):
    name: str = Field(
        description="The full name of the PhD applicant/scholar."
    )
    content: str = Field(
        description="A structured summary of the core content, research background, academic objectives, and proposed PhD research topic."
    )
    skills: List[str] = Field(
        description="A comprehensive list of technical, research, laboratory, programming, and analytical skills highlighted in the SOP."
    )

print("Pydantic schema successfully defined!")

Pydantic schema successfully defined!


In [9]:
from langchain_ollama import ChatOllama

# Initialize the model as shown in your configuration
llm = ChatOllama(
    model="gpt-oss:120b-cloud",
    temperature=0.0,
    base_url="http://127.0.0.1:11434"
)

# Force the Ollama model to emit JSON matching the Pydantic schema.
structured_llm = llm.with_structured_output(
    PhDScholarSOP,
    method="json_schema"
)

print("Ollama model initialized with JSON-schema structured output enabled.")

Ollama model initialized with JSON-schema structured output enabled.


In [5]:
print(llm.invoke("Hi"))

content='Hello! 👋 How can I help you today?' additional_kwargs={} response_metadata={'model': 'gpt-oss:120b-cloud', 'created_at': '2026-08-20T10:13:06.368508317Z', 'done': True, 'done_reason': 'stop', 'total_duration': 280546187, 'load_duration': None, 'prompt_eval_count': 68, 'prompt_eval_duration': None, 'eval_count': 46, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b-cloud', 'model_provider': 'ollama'} id='lc_run--01a01ea8-f6cc-72c1-8a50-6408cf815cf6-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 68, 'output_tokens': 46, 'total_tokens': 114}


In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert academic admissions reviewer. Extract the scholar's full name, "
        "core research content/objectives, and all academic/technical skills from the SOP. "
        "Return only the structured response. Use exactly these JSON keys and no alternatives: "
        "name, content, skills. The name and content values must be strings, and skills must "
        "be a list of strings. If information is missing, use 'Unknown' for strings and an "
        "empty list for skills."
    ),
    ("human", "Statement of Purpose:\n\n{sop_text}")
])

# Create the runnable chain
extraction_chain = prompt | structured_llm
print("Extraction chain ready.")

Extraction chain ready.


In [15]:
# Sample PhD SOP input text
sample_sop = """
Statement of Purpose

Applicant: Dr. / Mr. Ethan Vance
Target Program: Ph.D. in Computer Science (Artificial Intelligence & Robotics)

Having completed my Master of Science in Data Science at Stanford University, I am writing to express my strong interest in pursuing a Ph.D. under the supervision of Dr. Alistair Finch. My prior research focused on reinforcement learning for autonomous robotic manipulation. During my master's thesis, I developed novel policy-gradient optimization algorithms that reduced training latency by 35% in simulated environments. 

My technical foundation includes proficiency in Python, C++, PyTorch, ROS (Robot Operating System), and Gazebo simulation. In addition, I have hands-on experience in computer vision (OpenCV, YOLOv8), distributed model training using CUDA, and statistical hypothesis testing. In my doctoral studies, I aim to bridge sim-to-real domain gaps for multi-agent robotic swarms using transformer-based spatial representations.
"""

# Invoke the extraction chain
extracted_data: PhDScholarSOP = extraction_chain.invoke({"sop_text": extracted_content})

In [16]:
import json

# Access attributes directly
print(f"Candidate Name : {extracted_data.name}")
print(f"\nSkills Extracted ({len(extracted_data.skills)}):")
for skill in extracted_data.skills:
    print(f" - {skill}")

print("\n--- Core SOP Content ---")
print(extracted_data.content)

print("\n--- Raw JSON Dump ---")
print(json.dumps(extracted_data.model_dump(), indent=2))

Candidate Name : Hyungjoo Chae

Skills Extracted (24):
 - Large Language Models (LLMs)
 - Reinforcement Learning (RL)
 - RLHF (Reinforcement Learning from Human Feedback)
 - Self‑debugging of generated code
 - Code generation and compilation
 - Test‑case based reward functions
 - Knowledge‑grounded reasoning
 - Web navigation agents
 - World model simulation
 - Video‑based environment prediction
 - Behavior cloning
 - Reward modeling and pairwise preference collection
 - Policy training with RL
 - GUI interaction programming
 - Use of development tools (VSCode, terminals)
 - Algorithmic reasoning in language models
 - Conversational AI and dialogue systems
 - Natural Language Processing (NLP)
 - Evaluation environment design (e.g., Coffee‑gym)
 - Research and academic writing
 - Python programming
 - Machine learning framework proficiency (e.g., PyTorch, TensorFlow)
 - Data‑efficient learning
 - Safety in interactive AI systems

--- Core SOP Content ---
My research focuses on advancing

In [ ]:
import os
import json
import pymupdf  # PyMuPDF
from typing import List
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# ==========================================
# 1. CONSTANTS (Update these paths as needed)
# ==========================================
INPUT_PATH = r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\raw\phd\sop"
OUTPUT_PATH = r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\phd\sop"

# ==========================================
# 2. SCHEMA & LLM SETUP
# ==========================================
class PhDScholarSOP(BaseModel):
    name: str = Field(description="The full name of the PhD applicant/scholar.")
    content: str = Field(description="A structured summary of the core content, research background, academic objectives, and proposed PhD research topic.")
    skills: List[str] = Field(description="A comprehensive list of technical, research, laboratory, programming, and analytical skills highlighted in the SOP.")

# Initialize the model
llm = ChatOllama(
    model="gpt-oss:120b-cloud",
    temperature=0.0,
    base_url="http://127.0.0.1:11434"
)

# Force the Ollama model to emit JSON matching the Pydantic schema
structured_llm = llm.with_structured_output(
    PhDScholarSOP,
    method="json_schema"
)

# Define the prompt
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert academic admissions reviewer. Extract the scholar's full name, "
        "core research content/objectives, and all academic/technical skills from the SOP. "
        "Return only the structured response. Use exactly these JSON keys and no alternatives: "
        "name, content, skills. The name and content values must be strings, and skills must "
        "be a list of strings. If information is missing, use 'Unknown' for strings and an "
        "empty list for skills."
    ),
    ("human", "Statement of Purpose:\n\n{sop_text}")
])

# Create the runnable chain globally so it isn't recreated in the loop
extraction_chain = prompt | structured_llm


# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def extract_text_from_pdf(pdf_path: str) -> str:
    """Extracts text from a single PDF file."""
    try:
        doc = pymupdf.open(pdf_path)
        full_text = [page.get_text() for page in doc]
        return "\n".join(full_text)
    except Exception as e:
        print(f"Failed to read PDF {pdf_path}: {e}")
        return ""


# ==========================================
# 4. MAIN BATCH PROCESSING FUNCTION
# ==========================================
def process_sop_batches(input_dir: str, output_dir: str, batch_size: int = 10):
    """
    Scans the input directory for PDFs, extracts SOP data in batches using an LLM,
    and resumes from the last saved state without overwriting previous data.
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    output_file_path = os.path.join(output_dir, "extracted_sops.json")
    
    # --- RESUME LOGIC: Load existing data if it exists ---
    all_extracted_data = []
    processed_files = set()
    
    if os.path.exists(output_file_path):
        try:
            with open(output_file_path, "r", encoding="utf-8") as f:
                all_extracted_data = json.load(f)
                # Create a set of filenames we have already successfully processed
                processed_files = {record.get("source_file") for record in all_extracted_data if "source_file" in record}
            print(f"Resuming... Found {len(processed_files)} previously processed files in the JSON.")
        except json.JSONDecodeError:
            print("Warning: Existing JSON file is corrupted or empty. Starting fresh but preserving the file.")
    
    # Get a list of all PDF files in the input directory
    all_pdf_files = [f for f in os.listdir(input_dir) if f.lower().endswith(".pdf")]
    
    # Filter out the files that have already been processed
    pending_pdf_files = [
        os.path.join(input_dir, f) 
        for f in all_pdf_files 
        if f not in processed_files
    ]
    
    total_pending = len(pending_pdf_files)
    
    if total_pending == 0:
        print("All files in the directory have already been processed! Exiting.")
        return

    print(f"Found {len(all_pdf_files)} total PDF(s). {total_pending} remaining to process.")
    
    # Process the remaining files in chunks of 'batch_size'
    for i in range(0, total_pending, batch_size):
        batch_files = pending_pdf_files[i : i + batch_size]
        batch_number = (i // batch_size) + 1
        
        print(f"\n--- Processing Batch {batch_number} ({len(batch_files)} files) ---")
        
        for pdf_path in batch_files:
            file_name = os.path.basename(pdf_path)
            print(f"Extracting: {file_name}")
            
            # 1. Extract text from PDF
            sop_text = extract_text_from_pdf(pdf_path)
            if not sop_text.strip():
                print(f"   -> Warning: No text found in {file_name}. Skipping.")
                continue
            
            # 2. Run LLM Extraction
            try:
                extracted_data: PhDScholarSOP = extraction_chain.invoke({"sop_text": sop_text})
                
                # Convert Pydantic object to dict and append source filename
                record = extracted_data.model_dump()
                record["source_file"] = file_name
                
                all_extracted_data.append(record)
                print(f"   -> Success: Extracted data for {record.get('name', 'Unknown')}")
                
            except Exception as e:
                print(f"   -> Error during LLM extraction for {file_name}: {e}")
        
        # 3. Save accumulated data after every batch (overwrites with the full updated list)
        with open(output_file_path, "w", encoding="utf-8") as f:
            json.dump(all_extracted_data, f, indent=4)
            
        print(f"Batch {batch_number} complete. Progress saved to '{output_file_path}'.")

    print("\n==========================================")
    print(f"All batches processed! Total profiles extracted: {len(all_extracted_data)}")
    print(f"Final output saved to: {output_file_path}")
    print("==========================================")


# ==========================================
# 5. EXECUTION
# ==========================================
if __name__ == "__main__":
    process_sop_batches(
        input_dir=INPUT_PATH, 
        output_dir=OUTPUT_PATH, 
        batch_size=10
    )